### Embedding models

> https://docs.langchain.com/oss/python/integrations/text_embedding

> https://docs.langchain.com/oss/python/integrations/text_embedding/google_generative_ai

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# uv add langchain-google-genai

> https://ai.google.dev/gemini-api/docs/embeddings?hl=ko#python

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

[-0.023955047, 0.011876456, -0.0033613679, -0.0584139, 0.0015592978]

In [11]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

# 1. 실제 임베딩을 수행할 기본 모델 설정 (Google Gemini)
# 텍스트를 숫자로 변환하는 핵심 엔진입니다. 외부 API를 호출하므로 실행 시 비용과 시간이 소요됩니다.
underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. 로컬 파일 저장소 설정
# 임베딩된 결과(벡터 데이터)를 내 컴퓨터의 "./cache/" 폴더에 파일 형태로 저장하도록 지정합니다.
store = LocalFileStore("./cache/")

# 3. 캐시 지원 임베더(CacheBackedEmbeddings) 생성
# 실제 모델과 저장소를 연결하여 '캐싱' 기능을 제공하는 인터페이스입니다.
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,               # 실제 계산을 수행할 기본 모델
    store,                              # 계산된 벡터를 저장해둘 장소
    namespace = underlying_embeddings.model # 모델 이름별로 저장 구역을 분리 (모델 변경 시 데이터 충돌 방지)
)

# [작동 원리 요약]
# - 처음 실행 시: 구글 API를 통해 임베딩을 수행하고 결과를 './cache/'에 저장합니다.
# - 이후 재실행 시: 구글 API를 호출하지 않고 저장된 파일을 바로 읽어와서 속도가 비약적으로 향상됩니다.

### Vector stores

> https://docs.langchain.com/oss/python/integrations/vectorstores

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    splitted_documents,
    cached_embedder,
)

In [9]:
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"

In [10]:
results = vectorstore.similarity_search(query)
results

[Document(id='58afbb9d-dc0d-4003-bd70-60a4907f174c', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator': 'PDFium', 'Producer': 'PDFium'}, page_content='1.2 Validity of Collected Data\n본 연구에서는 의료기기 임상시험에 특화된 Private\n수집된 데이터셋은 의료기기 임상시험에 특화된\nLLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화\nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총\n용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로\n111,954페이지로 구성된 데이터는 의료기기 임상시험의\n구성된다. 각 단계는 의료기기 임상시험 분야의 특수성을\n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n반영하여 상호 유기적으로 작동하며, Figure 1와 같이 이\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수\n를 통해 해당 분야에서 최적의 성능을 달성하도록 설계되\n있는 다양한 시나리오를 반영하도록 설계되었다.\n었다.'),
 Document(id='e4905e25-9d62-4c4b-816c-53b35ad0876f', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator